# Practical worksheet: Discriminative vs Generative Models (2D)

In this worksheet you will build, step by step, two models for a simple 2D classification problem:

1. A **discriminative** model: learns how to separate classes (related to `p(y|x)`).
2. A **classic generative** model: learns how data are generated **per class** (related to `p(x|y)`), classifies using Bayes' rule, and can **generate new points**.

---

## Learning focus
You should be able to:
- Explain (in your own words) the difference between discriminative and generative modeling.
- Train a simple classifier and visualize the decision boundary.
- Fit a class-conditional Gaussian model, classify with Bayes, and sample new data.

---

## Allowed tools
- `numpy`, `matplotlib`
- `scikit-learn` (for train/test split and Logistic Regression)

Not allowed: deep generative models (VAE/GAN/Diffusion), or any pre-trained generative systems.

> Work cell-by-cell. Run often. If a plot looks wrong, fix earlier steps first.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2026)


## 1) Build a 2D dataset (two classes)

We will sample two classes from two 2D Gaussians:
- Class 0: `N(μ0, Σ0)`
- Class 1: `N(μ1, Σ1)`

### What should happen
- `X` should have shape `(n0+n1, 2)` (two features).
- `y` should contain labels `0` and `1`.
- A scatter plot should show two point clouds (possibly overlapping).

### Hints
- `μ` is a 2D vector, e.g. `[-1.0, 0.5]`.
- `Σ` is a **2x2 symmetric** matrix, e.g. `[[1.0, 0.3],[0.3, 1.2]]`.
- If the covariance is not valid, sampling may fail.


In [ ]:
def make_2d_dataset(n0=400, n1=400):
    # Choose means and covariances for each class.
    # Keep covariances symmetric (same off-diagonal entries).

    # TODO: define mean and covariance for class 0
    mu0 = np.array([____, ____])
    C0  = np.array([[____, ____],
                    [____, ____]])

    # TODO: define mean and covariance for class 1
    mu1 = np.array([____, ____])
    C1  = np.array([[____, ____],
                    [____, ____]])

    X0 = np.random.multivariate_normal(mu0, C0, size=n0)
    X1 = np.random.multivariate_normal(mu1, C1, size=n1)

    # Combine and label
    X = np.vstack([X0, X1])
    y = np.hstack([np.zeros(n0, dtype=int), np.ones(n1, dtype=int)])

    return X, y

X, y = make_2d_dataset()
print("X shape:", X.shape)
print("label counts:", np.unique(y, return_counts=True))


### Quick check (expected)
- `X shape: (800, 2)` if you used the default `n0=n1=400`.
- Label counts should show two numbers close to 400 each.


## 2) Visualize the dataset

Plot the two classes with different colors/labels.
A reasonable plot shows two clusters that may overlap.

If you see only one color, check your boolean masks (`y==0`, `y==1`).


In [ ]:
plt.figure()

plt.scatter(X[y==0,0], X[y==0,1], s=12, alpha=0.6, label="class 0")
plt.scatter(X[y==1,0], X[y==1,1], s=12, alpha=0.6, label="class 1")
plt.title("2D dataset")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend()
plt.show()


## 3) Train / test split

To evaluate how well our models generalize, we split the dataset into:

- a **training set**, used to fit the models
- a **test set**, used only for evaluation

We use a **stratified split**, so that the proportion of each class
is preserved in both sets.

This is important when class frequencies are not exactly equal.



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=2026,
    stratify=y
)

print("train:", X_train.shape, "test:", X_test.shape)


## 4) Discriminative model: Logistic Regression

Logistic Regression is a **discriminative model**.

It learns a decision function that directly predicts the label from the input:
- Conceptually related to learning $p(y \mid x) $
- It does NOT model how the data are generated

Key idea:
- The model learns a decision boundary (typically linear in feature space)
- It focuses only on separating classes

After training:
- Evaluate performance on the test set
- Later we will visualize the learned boundary


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# TODO: instantiate and train
disc = LogisticRegression()
disc.fit(____, ____)

# TODO: predict and evaluate
pred_disc = disc.predict(____)
print("Accuracy (discriminative):", accuracy_score(y_test, pred_disc))


### Expected result (example)
Your accuracy should usually be **well above chance (0.5)**.  
Typical outcomes are around **0.80–0.95**, depending on your dataset overlap.


## 5) Visualizing the decision boundary

To better understand what the model learned, we visualize its decision boundary.

We:
1. Create a grid covering the 2D feature space.
2. Predict the class for every point in the grid.
3. Plot the predicted regions as a background.
4. Overlay the original data points.

This allows us to see:
- The shape of the decision boundary
- Whether it is linear or curved
- How well it separates the classes


Make sure you:
- call `model.predict(grid)`
- reshape predictions to `xx.shape`


In [ ]:
def plot_boundary(model, X, y, title=""):
    x_min, x_max = X[:,0].min()-1.0, X[:,0].max()+1.0
    y_min, y_max = X[:,1].min()-1.0, X[:,1].max()+1.0

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    # TODO: predict class for each grid point
    zz = model.predict(____)
    
    # TODO: reshape predictions to match grid shape
    zz = zz.reshape(____)

    plt.figure()
    plt.contourf(xx, yy, zz, alpha=0.25)
    plt.scatter(X[y==0,0], X[y==0,1],s=12, alpha=0.6, label="class 0")
    plt.scatter(X[y==1,0], X[y==1,1],s=12, alpha=0.6, label="class 1")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.show()

plot_boundary(disc, X, y,title="Decision boundary: Discriminative (Logistic Regression)")


## 6) Generative classification using Bayes’ rule

### Goal

So far, we have learned **how the data are generated inside each class**.  
For each class $ c $, we model:

$
p(x \mid y=c) = \mathcal{N}(x \mid \mu_c, \Sigma_c)
$

This describes **how points look within a class**, but it is **not yet a classifier**.

To assign a class label to a new point $ x $, we must use **Bayes’ rule**.

---

### Bayes’ rule for classification

Bayes’ rule states:

$
p(y=c \mid x) = \frac{p(x \mid y=c)\,p(y=c)}{p(x)}
$

where:
- $ p(x \mid y=c) $ is the **class-conditional likelihood**
- $ p(y=c) $ is the **prior probability** of class $ c $
- $ p(x) $ is the evidence (same for all classes)

Since $ p(x) $ does **not depend on the class**, it can be ignored when comparing classes:

$
p(y=c \mid x) \propto p(x \mid y=c)\,p(y=c)
$

---

### Using log-probabilities

In practice, probabilities can become extremely small.  
To avoid numerical underflow, we work in **log-space**:

$
\log p(y=c \mid x) = \log p(x \mid y=c) + \log p(y=c) + \text{constant}
$

The constant is identical for all classes and can be ignored.

We therefore define a **classification score**:

$
\text{score}_c(x) = \log p(x \mid y=c) + \log p(y=c)
$

The predicted class is the one with the highest score:

$
\hat{y} = \arg\max_c \; \text{score}_c(x)
$

---

### What this means conceptually

- The model **does not learn a decision boundary directly**
- It learns a **probability distribution over data**
- Classification **emerges** from the generative assumptions
- The same model can be used to:
  - classify
  - generate new data
  - compute likelihoods

---

### Pseudocode: Generative classification with Bayes’ rule


For a new input point x:

    for each class c:
        compute log_likelihood = log p(x | y = c)
        compute log_prior      = log p(y = c)
        score[c] = log_likelihood + log_prior

    return the class c with the maximum score



---

### Notes

- Class priors $ p(y=c) $ are usually estimated from the training data
- If priors are equal, classification depends only on $ p(x \mid y) $
- Different covariance matrices lead to **curved decision boundaries**
- If the Gaussian assumption is wrong, classification performance may degrade



In [ ]:
def fit_gaussian_per_class(X, y):
    params = {}
    classes = np.unique(y)

    for c in classes:
        Xc = X[y == c]

        # TODO: compute mean vector (shape: (2,))
        mu = ____.mean(axis=0)

        # TODO: compute covariance matrix (shape: (2,2))
        # Hint: np.cov expects variables in rows -> try np.cov(Xc.T)
        cov = np.cov(____)

        params[int(c)] = (mu, cov)

    return params

gen_params = fit_gaussian_per_class(X_train, y_train)
gen_params


## 7) Classify with Bayes (log-scores)

Generative classification uses:

`p(y|x) ∝ p(x|y) p(y)`

In log form:

`score_c(x) = log p(x|y=c) + log p(y=c)`

You will implement:
- `log_gaussian_pdf` (multivariate Gaussian log-density)
- `predict_generative` (choose class with highest score)

Numerical stability:
- regularize covariance: `cov = cov + eps * I`
- use `np.linalg.slogdet` for `log|Σ|`

### Expected result (example)
- Accuracy should again be **above chance**.
- Depending on your dataset choice, the generative model may be slightly worse or comparable to Logistic Regression.


In [ ]:
def log_gaussian_pdf(X, mu, cov, eps=1e-6):
    # X: (N,2)
    # Regularize covariance to avoid singular matrices
    cov = ____ + eps*np.eye(2)

    # TODO: inverse and log-determinant
    inv = np.linalg.inv(____)
    sign, logdet = np.linalg.slogdet(____)

    diff = X - mu

    # TODO: Mahalanobis term: (x-mu)^T inv (x-mu)
    quad = np.sum((___ @ inv) * ___, axis=1)

    d = X.shape[1]
    return -0.5 * (quad + logdet + d*np.log(2*np.pi))

def predict_generative(X, params, priors):
    classes = sorted(params.keys())
    scores = []

    for c in classes:
        mu, cov = params[c]
        # TODO: score = log p(x|c) + log p(c)     -- (using priors)
        s = log_gaussian_pdf(X, mu, cov) + np.log(____)
        scores.append(s)

    scores = np.vstack(scores).T  # (N, K)
    return np.argmax(scores, axis=1)

priors = {
    0: np.mean(y_train == 0),
    1: np.mean(y_train == 1),
}

pred_gen = predict_generative(X_test, gen_params, priors)
print("Accuracy (generative):", accuracy_score(y_test, pred_gen))


## 8) Visualize generative decision boundary

We reuse the same `plot_boundary` by creating a wrapper that provides `.predict(X)`.


In [ ]:
class GaussianGenerativeClassifier:
    def __init__(self, params, priors):
        self.params = params
        self.priors = priors

    def predict(self, X):
        # TODO: call predict_generative
        return predict_generative(____, ____, ____)

gen_model = GaussianGenerativeClassifier(gen_params, priors)
plot_boundary(gen_model, X, y, title="Decision boundary: Generative (class-conditional Gaussians)")


## 9) Sampling: generate new points `x ~ p(x|y)`

This is the “generative” capability:
- Once `μ_c` and `Σ_c` are estimated, we can generate new points from each class distribution.

A good plot will show generated points overlapping the original training clouds.


In [ ]:
def sample_class(c, n, params):
    mu, cov = params[c]
    # TODO: sample n points from N(mu, cov)
    return np.random.multivariate_normal(____, ____, size=____)

X0_new = sample_class(0, 200, gen_params)
X1_new = sample_class(1, 200, gen_params)

plt.figure()
plt.scatter(X_train[y_train==0,0], X_train[y_train==0,1], s=10, alpha=0.2, label="train class 0")
plt.scatter(X_train[y_train==1,0], X_train[y_train==1,1], s=10, alpha=0.2, label="train class 1")
plt.scatter(X0_new[:,0], X0_new[:,1], s=14, alpha=0.8, label="generated class 0")
plt.scatter(X1_new[:,0], X1_new[:,1], s=14, alpha=0.8, label="generated class 1")
plt.title("Generated samples from the generative model")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend()
plt.show()


## 10) Short reflection (write short answers)

1. Which model directly learns `p(y|x)` (or an equivalent boundary)?
2. Which model estimates `p(x|y)` and then uses Bayes to obtain `p(y|x)`?
3. Why can the generative model generate new points while Logistic Regression cannot?

---

### Extra mini-experiment
Go back to the start, change your dataset parameters (`μ` and `Σ`) and observe how:
- overlap changes
- decision boundaries change
- accuracies change


Answers:

1.

2.

3.